In [0]:
# ============================================================
# BRONZE LAYER SETUP
# IT Financial Lakehouse - Enterprise Spend Analytics
# ============================================================

# Define medallion layer paths
BRONZE_PATH = "/it_finance_lakehouse/bronze"
SILVER_PATH = "/it_finance_lakehouse/silver"
GOLD_PATH   = "/it_finance_lakehouse/gold"

# Create databases for each layer
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_it_finance")
spark.sql("CREATE DATABASE IF NOT EXISTS silver_it_finance")
spark.sql("CREATE DATABASE IF NOT EXISTS gold_it_finance")

print("✅ Medallion layer databases created successfully")
print(f"   Bronze: {BRONZE_PATH}")
print(f"   Silver: {SILVER_PATH}")
print(f"   Gold:   {GOLD_PATH}")





✅ Medallion layer databases created successfully
   Bronze: /it_finance_lakehouse/bronze
   Silver: /it_finance_lakehouse/silver
   Gold:   /it_finance_lakehouse/gold


In [0]:
# ============================================================
# GENERATE SYNTHETIC IT FINANCE SOURCE DATA
# Simulates: SAP GL Extract + Vendor Spend + Cost Center Data
# ============================================================

from pyspark.sql import Row
from datetime import date

# --- Cost Center Reference Data ---
cost_centers = [
    Row(cost_center_id="CC-4010", cost_center_name="Cloud Operations", department="Infrastructure"),
    Row(cost_center_id="CC-4020", cost_center_name="Cybersecurity", department="Infrastructure"),
    Row(cost_center_id="CC-4030", cost_center_name="Enterprise Applications", department="Corporate IT"),
    Row(cost_center_id="CC-4040", cost_center_name="Data & Analytics", department="Corporate IT"),
    Row(cost_center_id="CC-4050", cost_center_name="Service Desk", department="End User Services"),
]

# --- Vendor Spend Transactions (simulates SAP GL extract) ---
transactions = [
    Row(transaction_id="TXN-001", vendor="AWS", cost_center_id="CC-4010", amount=142500.00, fiscal_period="2026-01", spend_category="Cloud Infrastructure", forecast_cycle="3+9"),
    Row(transaction_id="TXN-002", vendor="Microsoft Azure", cost_center_id="CC-4010", amount=98200.00, fiscal_period="2026-01", spend_category="Cloud Infrastructure", forecast_cycle="3+9"),
    Row(transaction_id="TXN-003", vendor="CrowdStrike", cost_center_id="CC-4020", amount=54000.00, fiscal_period="2026-01", spend_category="Security Software", forecast_cycle="3+9"),
    Row(transaction_id="TXN-004", vendor="ServiceNow", cost_center_id="CC-4030", amount=76800.00, fiscal_period="2026-01", spend_category="ITSM Platform", forecast_cycle="3+9"),
    Row(transaction_id="TXN-005", vendor="Databricks", cost_center_id="CC-4040", amount=38400.00, fiscal_period="2026-01", spend_category="Data Platform", forecast_cycle="3+9"),
    Row(transaction_id="TXN-006", vendor="AWS", cost_center_id="CC-4010", amount=158900.00, fiscal_period="2026-02", spend_category="Cloud Infrastructure", forecast_cycle="3+9"),
    Row(transaction_id="TXN-007", vendor="CrowdStrike", cost_center_id="CC-4020", amount=54000.00, fiscal_period="2026-02", spend_category="Security Software", forecast_cycle="3+9"),
    Row(transaction_id="TXN-008", vendor="Salesforce", cost_center_id="CC-4030", amount=29500.00, fiscal_period="2026-02", spend_category="CRM Platform", forecast_cycle="3+9"),
    Row(transaction_id="TXN-009", vendor="Databricks", cost_center_id="CC-4040", amount=41200.00, fiscal_period="2026-02", spend_category="Data Platform", forecast_cycle="3+9"),
    Row(transaction_id="TXN-010", vendor="Microsoft Azure", cost_center_id="CC-4010", amount=103400.00, fiscal_period="2026-03", spend_category="Cloud Infrastructure", forecast_cycle="3+9"),
]

# Create DataFrames
df_cost_centers = spark.createDataFrame(cost_centers)
df_transactions = spark.createDataFrame(transactions)

print(f"✅ Cost Centers loaded: {df_cost_centers.count()} records")
print(f"✅ Transactions loaded: {df_transactions.count()} records")
df_transactions.show(truncate=False)

✅ Cost Centers loaded: 5 records
✅ Transactions loaded: 10 records
+--------------+---------------+--------------+--------+-------------+--------------------+--------------+
|transaction_id|vendor         |cost_center_id|amount  |fiscal_period|spend_category      |forecast_cycle|
+--------------+---------------+--------------+--------+-------------+--------------------+--------------+
|TXN-001       |AWS            |CC-4010       |142500.0|2026-01      |Cloud Infrastructure|3+9           |
|TXN-002       |Microsoft Azure|CC-4010       |98200.0 |2026-01      |Cloud Infrastructure|3+9           |
|TXN-003       |CrowdStrike    |CC-4020       |54000.0 |2026-01      |Security Software   |3+9           |
|TXN-004       |ServiceNow     |CC-4030       |76800.0 |2026-01      |ITSM Platform       |3+9           |
|TXN-005       |Databricks     |CC-4040       |38400.0 |2026-01      |Data Platform       |3+9           |
|TXN-006       |AWS            |CC-4010       |158900.0|2026-02      |Cloud I

In [0]:
# ============================================================
# WRITE BRONZE LAYER TO DELTA
# Append-only, raw ingestion — no transformations
# ============================================================

# Write cost centers to Bronze Delta table
df_cost_centers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_it_finance.raw_cost_centers")

# Write transactions to Bronze Delta table
df_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_it_finance.raw_vendor_transactions")

print("✅ Bronze Delta tables written successfully")
print("   → bronze_it_finance.raw_cost_centers")
print("   → bronze_it_finance.raw_vendor_transactions")

✅ Bronze Delta tables written successfully
   → bronze_it_finance.raw_cost_centers
   → bronze_it_finance.raw_vendor_transactions


In [0]:
# ============================================================
# SILVER LAYER - Clean, Conform, Surrogate Keys
# Reads from Bronze, applies business rules, writes to Silver
# ============================================================

from pyspark.sql.functions import col, trim, md5, concat_ws, current_timestamp, to_date, lit, concat

# Read from Bronze
df_bronze_txn = spark.table("bronze_it_finance.raw_vendor_transactions")
df_bronze_cc  = spark.table("bronze_it_finance.raw_cost_centers")

# --- Clean & Conform Transactions ---
df_silver_txn = df_bronze_txn \
    .withColumn("vendor", trim(col("vendor"))) \
    .withColumn("cost_center_id", trim(col("cost_center_id"))) \
    .withColumn("fiscal_period", to_date(concat(col("fiscal_period"), lit("-01")), "yyyy-MM-dd")) \
    .withColumn("amount", col("amount").cast("decimal(18,2)")) \
    .withColumn("spend_category", trim(col("spend_category"))) \
    .withColumn("forecast_cycle", trim(col("forecast_cycle"))) \
    .withColumn("surrogate_key", md5(concat_ws("|", col("transaction_id"), col("vendor")))) \
    .withColumn("dw_insert_ts", current_timestamp()) \
    .dropDuplicates(["transaction_id"])

# --- Clean & Conform Cost Centers ---
df_silver_cc = df_bronze_cc \
    .withColumn("cost_center_id", trim(col("cost_center_id"))) \
    .withColumn("cost_center_name", trim(col("cost_center_name"))) \
    .withColumn("department", trim(col("department"))) \
    .withColumn("surrogate_key", md5(concat_ws("|", col("cost_center_id"), col("cost_center_name")))) \
    .withColumn("dw_insert_ts", current_timestamp()) \
    .dropDuplicates(["cost_center_id"])

# Write to Silver
df_silver_txn.write.format("delta").mode("overwrite").saveAsTable("silver_it_finance.clean_vendor_transactions")
df_silver_cc.write.format("delta").mode("overwrite").saveAsTable("silver_it_finance.clean_cost_centers")

print("✅ Silver layer written successfully")
print(f"   → silver_it_finance.clean_vendor_transactions: {df_silver_txn.count()} records")
print(f"   → silver_it_finance.clean_cost_centers: {df_silver_cc.count()} records")
df_silver_txn.show(truncate=False)

✅ Silver layer written successfully
   → silver_it_finance.clean_vendor_transactions: 10 records
   → silver_it_finance.clean_cost_centers: 5 records
+--------------+---------------+--------------+---------+-------------+--------------------+--------------+--------------------------------+--------------------------+
|transaction_id|vendor         |cost_center_id|amount   |fiscal_period|spend_category      |forecast_cycle|surrogate_key                   |dw_insert_ts              |
+--------------+---------------+--------------+---------+-------------+--------------------+--------------+--------------------------------+--------------------------+
|TXN-001       |AWS            |CC-4010       |142500.00|2026-01-01   |Cloud Infrastructure|3+9           |1d70def941ce50a1c33cb00a8fb1e0a1|2026-05-24 02:46:26.396617|
|TXN-002       |Microsoft Azure|CC-4010       |98200.00 |2026-01-01   |Cloud Infrastructure|3+9           |8c1a5fe26c1f080231139b04f8001ff6|2026-05-24 02:46:26.396617|
|TXN-003  

In [0]:
# ============================================================
# GOLD LAYER - Kimball Star Schema
# fact_it_spend_monthly + dim_vendor + dim_cost_center
# ============================================================

from pyspark.sql.functions import col, md5, concat_ws, current_timestamp, sum as spark_sum, month, year

# Read from Silver
df_txn = spark.table("silver_it_finance.clean_vendor_transactions")
df_cc   = spark.table("silver_it_finance.clean_cost_centers")

# --- DIM: Vendor ---
df_dim_vendor = df_txn.select("vendor", "spend_category").distinct() \
    .withColumn("vendor_key", md5(concat_ws("|", col("vendor"), col("spend_category")))) \
    .withColumn("dw_insert_ts", current_timestamp())

# --- DIM: Cost Center ---
df_dim_cost_center = df_cc.select(
    col("surrogate_key").alias("cost_center_key"),
    col("cost_center_id"),
    col("cost_center_name"),
    col("department"),
    col("dw_insert_ts")
)

# --- FACT: Monthly IT Spend ---
df_fact = df_txn \
    .join(df_dim_vendor, on=["vendor", "spend_category"], how="left") \
    .join(df_cc.select(col("cost_center_id"), col("surrogate_key").alias("cost_center_key")), on="cost_center_id", how="left") \
    .groupBy("vendor_key", "cost_center_key", "fiscal_period", "forecast_cycle") \
    .agg(spark_sum("amount").alias("total_spend")) \
    .withColumn("fiscal_year", year(col("fiscal_period"))) \
    .withColumn("fiscal_month", month(col("fiscal_period"))) \
    .withColumn("dw_insert_ts", current_timestamp())

# Write Gold tables
df_dim_vendor.write.format("delta").mode("overwrite").saveAsTable("gold_it_finance.dim_vendor")
df_dim_cost_center.write.format("delta").mode("overwrite").saveAsTable("gold_it_finance.dim_cost_center")
df_fact.write.format("delta").mode("overwrite").saveAsTable("gold_it_finance.fact_it_spend_monthly")

print("✅ Gold layer written successfully")
print(f"   → gold_it_finance.dim_vendor: {df_dim_vendor.count()} records")
print(f"   → gold_it_finance.dim_cost_center: {df_dim_cost_center.count()} records")
print(f"   → gold_it_finance.fact_it_spend_monthly: {df_fact.count()} records")
df_fact.show(truncate=False)

✅ Gold layer written successfully
   → gold_it_finance.dim_vendor: 6 records
   → gold_it_finance.dim_cost_center: 5 records
   → gold_it_finance.fact_it_spend_monthly: 10 records
+--------------------------------+--------------------------------+-------------+--------------+-----------+-----------+------------+--------------------------+
|vendor_key                      |cost_center_key                 |fiscal_period|forecast_cycle|total_spend|fiscal_year|fiscal_month|dw_insert_ts              |
+--------------------------------+--------------------------------+-------------+--------------+-----------+-----------+------------+--------------------------+
|b38e3be340cc37eeee11c74835f75c0b|63c0a02f1ed8b4831a41207860c535a6|2026-02-01   |3+9           |29500.00   |2026       |2           |2026-05-24 02:48:19.896719|
|47d0db72010c0a23c6e944133853049e|b8df916c2767e3566c2329ca1781d553|2026-02-01   |3+9           |54000.00   |2026       |2           |2026-05-24 02:48:19.896719|
|94cee2b48d8bbe